# Agents: a hands-on implementation

**Track:** Agents · **Guide:** [`050-agents.md`](../guides/concepts/050-agents.md)

An agent combines a model with a bounded set of tools and state to pursue a goal over multiple steps.

> **Intuition:** A careful intern can consult approved tools, but needs a clear task and limits on what they may change.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Agents]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** An operations agent explains a failed job using logs but cannot restart production without approval.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Define a goal and least-privilege tools

An agent has a model or policy, task state, and explicitly permitted tools. Start with read-only operations.


In [ ]:
from collections.abc import Callable
from dataclasses import dataclass

documents = {"returns": "Unused items may be returned within 30 days."}

def search_policy(topic: str) -> str:
    return documents.get(topic, "No matching policy found.")

def calculator(expression: str) -> float:
    allowed = {"2 + 2": 4.0, "10 * 3": 30.0}
    if expression not in allowed:
        raise ValueError("expression is not in this demo's allowlist")
    return allowed[expression]

tools: dict[str, Callable] = {
    "search_policy": search_policy,
    "calculator": calculator,
}

## 2. Parse a plan and validate tool calls

A real agent asks a model to select a tool. This offline plan demonstrates typed dispatch and rejects tools outside the allowlist.


In [ ]:
@dataclass
class ToolCall:
    name: str
    arguments: dict[str, str]

def execute_tool(call: ToolCall):
    if call.name not in tools:
        raise PermissionError(f"tool is not permitted: {call.name}")
    return tools[call.name](**call.arguments)

planned_call = ToolCall("search_policy", {"topic": "returns"})
observation = execute_tool(planned_call)
print("observation:", observation)

## 3. Produce a bounded, evidence-linked result

The agent should report the tool result, not claim more than its evidence supports. Side effects require separate authorization.


In [ ]:
answer = {
    "question": "What is the return window?",
    "evidence": observation,
    "answer": observation,
    "side_effects_performed": False,
}
print(answer)

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/050-agents.md) and the
[learning roadmap](../guides/ROADMAP.md).
